In [1]:
import scanpy as sc
import scrublet as scr
import numpy as np
import pandas as pd
import sctk
import muon as mu
from datetime import datetime
import os
import matplotlib.pyplot as plt
import seaborn as sns
import json

2024-11-26 10:31:25.269632: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-26 10:31:25.269681: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-26 10:31:25.269705: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-26 10:31:25.918620: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [40]:
atribute = 'Elementaite2021_E-MTAB-9543_QC'
adata = sc.read_h5ad('gut_hs_Elementaite2021_E-MTAB-9543_AM_05112024_113505_raw.h5ad')
adata

AnnData object with n_obs × n_vars = 139345920 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[sex]', 'Characteristics[individual]', 'Characteristics[cell type]', 'Characteristics[organism part]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library

In [41]:
sc.pp.filter_cells(adata, min_genes=100)

In [42]:
adata

AnnData object with n_obs × n_vars = 239919 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[sex]', 'Characteristics[individual]', 'Characteristics[cell type]', 'Characteristics[organism part]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library co

In [43]:
# take random 5% of cells
np.random.seed(42)
adata = adata[np.random.choice(adata.obs.index, int(0.1*adata.n_obs), replace=False), :]

In [44]:
adata

View of AnnData object with n_obs × n_vars = 23991 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[sex]', 'Characteristics[individual]', 'Characteristics[cell type]', 'Characteristics[organism part]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[lib

In [45]:
#delete 'Source Name' with less than 30 cells
adata = adata[~adata.obs['Source Name'].isin(adata.obs['Source Name'].value_counts()[adata.obs['Source Name'].value_counts() < 100].index), :]

In [46]:
adata.obs['Source Name'].value_counts()

Source Name
Human_colon_16S8117828    3267
Human_colon_16S8001903    1429
Human_colon_16S8117831    1352
Human_colon_16S8117830    1252
Human_colon_16S8000513    1166
Human_colon_16S8123921    1024
Human_colon_16S8117829     928
Human_colon_16S8123911     726
Human_colon_16S8002629     706
Human_colon_16S8001865     643
Human_colon_16S8002630     640
Human_colon_16S8002624     608
Human_colon_16S8002627     587
Human_colon_16S8002626     571
Human_colon_16S8002628     451
Human_colon_16S8001879     438
Human_colon_16S8002623     413
Human_colon_16S8159191     377
Human_colon_16S8123916     365
Human_colon_16S8001885     345
Human_colon_16S8123917     342
Human_colon_16S8001907     301
Human_colon_16S8123913     268
Human_colon_16S8001869     261
Human_colon_16S8159193     253
Human_colon_16S8001871     240
Human_colon_16S8123915     229
Human_colon_16S8001863     218
Human_colon_16S8123910     213
Human_colon_16S8001873     211
Human_colon_16S8123919     200
Human_colon_16S8000481     

In [48]:
sample_names = adata.obs['Source Name'].unique()

for sample_name in sample_names:
    mask = adata.obs['Source Name'] == sample_name
    sample_adata = adata[mask].copy()

    scrub = scr.Scrublet(sample_adata.X)

    sample_adata.obs['doublet_scores'], sample_adata.obs['predicted_doublets'] = scrub.scrub_doublets()

    adata.obs.loc[mask, 'doublet_scores'] = sample_adata.obs['doublet_scores']
    adata.obs.loc[mask, 'predicted_doublets'] = sample_adata.obs['predicted_doublets']


Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.40
Detected doublet rate = 1.4%
Estimated detectable doublet fraction = 8.1%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 16.9%
Elapsed time: 0.2 seconds
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.45
Detected doublet rate = 0.7%
Estimated detectable doublet fraction = 7.5%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 8.8%
Elapsed time: 0.2 seconds
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.42
Detected doublet rate = 1.2%
Estimated detectable doublet fraction = 1.2%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 100.0%
Elapsed time: 0.1 seconds
Preprocessing...
Simulating double

In [49]:
sctk.calculate_qc(adata)

In [50]:
sctk.cellwise_qc(adata)

n_counts: [980.2248477133548, 58761.00390625], 5035/23476 passed
n_genes: [99.9999893018715, 6007.00244140625], 23476/23476 passed
percent_mito: [0.0, 21.275509326556328], 10099/23476 passed
percent_ribo: [0.0, 14.892585044386555], 23231/23476 passed
percent_hb: [0.0, 0.0038057640634140966], 22295/23476 passed
832/23476 pass


In [51]:
metrics_list = ["log1p_n_counts", "log1p_n_genes", "percent_mito", "percent_ribo", "percent_hb"]
sctk.generate_qc_clusters(adata, metrics = metrics_list)

In [52]:
sctk.clusterwise_qc(adata)

In [53]:
sctk.multi_resolution_cluster_qc(adata, metrics = metrics_list)

Best overlap found for resolution 0.9


In [54]:
project = 'gut'
species = 'hs'
name = 'AM'
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')
counts = 'raw'

In [55]:
print(type(adata.uns['scautoqc_ranges']['low']))
print(adata.uns['scautoqc_ranges']['low'])

<class 'pandas.core.series.Series'>
n_counts        980.224848
n_genes          99.999989
percent_mito           0.0
percent_ribo           0.0
percent_hb             0.0
Name: low, dtype: object


In [56]:
adata

AnnData object with n_obs × n_vars = 23476 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[sex]', 'Characteristics[individual]', 'Characteristics[cell type]', 'Characteristics[organism part]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library con

In [58]:
adata.uns['processing_history']

{'step': 'create raw anndata after mapping, no filtering',
 'timestamp': '05112024_113505'}

In [60]:
current_history = adata.uns['processing_history']
new_history = [
     json.dumps(current_history),
     json.dumps({
          'timestamp': timestamp,
          'step': 'filtered cells based on consensus qc generated with sctk (default parameters), added sex covariates and cell cycle phase'
          })
        ]
adata.uns['processing_history'] = new_history

In [61]:
adata.uns['processing_history']

['{"step": "create raw anndata after mapping, no filtering", "timestamp": "05112024_113505"}',
 '{"timestamp": "26112024_114453", "step": "filtered cells based on consensus qc generated with sctk (default parameters), added sex covariates and cell cycle phase"}']

In [63]:
# Convert non-string objects to strings
adata.obs['predicted_doublets'] = adata.obs['predicted_doublets'].astype(str)

# Convert the Series to a dictionary and serialize it to a JSON string
adata.uns['scautoqc_ranges']['low'] = json.dumps(adata.uns['scautoqc_ranges']['low'].to_dict())
adata.uns['scautoqc_ranges']['high'] = json.dumps(adata.uns['scautoqc_ranges']['high'].to_dict())

adata.write_h5ad(f"Elementaite_2021/{project}_{species}_{atribute}_{name}_{timestamp}_{counts}_demo.h5ad")